# Caching API responses
Use this notebook to populate `RESPONSE_CACHE_FILE` with responses from that API that can later be used in the training loop

In [21]:
import io
import os
from asyncio import sleep

from PIL import Image
import pandas as pd

from homework_3.functions import parse_img, get_cache, query_api, set_cache, api_response_to_tensor

In [2]:
def cache_img(img: Image.Image) -> None:
    img_base64, img_hash = parse_img(img)
    if get_cache(img_hash):
        return
    api_response = query_api(img_base64)
    set_cache(img_hash, api_response)
    print(f"Cached {img_hash}")

In [3]:
dir_path = "../img"
images = [Image.open(f"{dir_path}/{p}") for p in os.listdir(dir_path)]

In [4]:
for i in images:
    cache_img(i)
print(f"Cached all images in {dir_path}")

Cached all images in ../img


In [36]:
try:
    df = pd.read_parquet("../datasets/cifar100_train_0:100.parquet")
    series = df.img[:100]
except FileNotFoundError:
    splits = {'train': 'cifar100/train-00000-of-00001.parquet', 'test': 'cifar100/test-00000-of-00001.parquet'}
    df = pd.read_parquet("hf://datasets/uoft-cs/cifar100/" + splits["train"])
    df.to_parquet("../datasets/cifar100_train_0:100.parquet")
    series = df.img[:100]

for i in series:
    image = Image.open(io.BytesIO(i['bytes']))
    cache_img(image)

In [32]:
responses = pd.read_json('../cache/responses.json')
for resp in responses.to_dict().values():
    for value in api_response_to_tensor(resp)[2:]:
        if value > 0.97:
            print(resp)

{'classes': {'class1': 0.6121724843978881, 'class2': 0.38782757520675604, 'class3': 0.970006227493286, 'class4': 0.029993832111358}}
{'classes': {'class1': 0.116222590208053, 'class2': 0.883777379989624, 'class3': 0.024060405790805, 'class4': 0.9759396314620971}}
